# 🤖 Бот «Лилит» ЦЕЛИКОМ на Google Colab (бесплатно, без карты)

Этот ноутбук запускает **весь бот**: разговор (LLM), голос (Piper), картинки (ComfyUI на GPU T4) — всё в одном месте. Нужен только **Google-аккаунт** (карта НЕ нужна).

## Как пользоваться (3 минуты):
1. Откройте [colab.research.google.com](https://colab.research.google.com) → **Файл → Загрузить блокнот** → выберите этот файл.
2. Меню **Среда выполнения → Сменить среду выполнения** → **T4 GPU** → **Сохранить**.
3. Запустите все ячейки по очереди (Shift+Enter) или **Среда выполнения → Выполнить всё**.
4. Когда спросит токен — вставьте токен от @BotFather.
5. Готово! Бот работает. Идите в Telegram и общайтесь.

## ⚠️ Честно про ограничения:
- Сессия Colab живёт **несколько часов** (бесплатно), потом прерывается — запустите ноутбук заново (2 минуты).
- При бездействии ~90 минут сессия отключается сама — бот не отвечает, пока не перезапустите.
- Это не 24/7, но **бесплатно и без карты** — лучший вариант без банковской карты.
- Когда закончили — **Среда выполнения → Прервать**, чтобы не тратить лимит GPU.

In [ ]:
# ⚡ АВТОЗАПУСК: ключи запоминаются, всё остальное делает ноутбук.
# Первый раз: введите ключи — сохранятся в секреты Colab.
# Дальше: просто «Выполнить всё» — ключи подхватятся сами.
import os, sys, getpass, time
try:
    from google.colab import userdata
    TG = userdata.get('LILITH_TELEGRAM_TOKEN') or ''
    CV = userdata.get('LILITH_CIVITAI_TOKEN') or ''
except Exception:
    TG = CV = ''
if not TG:
    TG = getpass.getpass('Токен Telegram от @BotFather: ').strip()
    try:
        userdata.set('LILITH_TELEGRAM_TOKEN', TG)
    except Exception:
        pass
if not CV:
    CV = getpass.getpass('API-ключ civitai.com (Enter чтобы пропустить): ').strip()
    if CV:
        try:
            userdata.set('LILITH_CIVITAI_TOKEN', CV)
        except Exception:
            pass
# OpenRouter НЕ нужен: языковая модель работает ЛОКАЛЬНО (Ollama, ячейка 3).
# Вам нужен только токен Telegram (обязательно) и ключ civitai (по желанию).
open('/content/.lilith_keys', 'w').write(f'{TG}\n{CV}')
print('✅ Ключи готовы:', 'Telegram ✓' if TG else 'Telegram ✗', '|', 'Civitai ✓' if CV else 'Civitai —')


In [ ]:
# 0. Проверяем GPU и память
!nvidia-smi --query-gpu=name,memory.total --format=csv
import psutil
print(f"RAM: {psutil.virtual_memory().total // (1024**3)} ГБ")

In [ ]:
# 1. Токен бота — автосохранение в секреты Colab (заполняется один раз)
import getpass, os
try:
    from google.colab import userdata
    TOKEN = userdata.get('LILITH_TELEGRAM_TOKEN')
    if TOKEN:
        print('✅ Токен найден в секретах Colab')
except Exception:
    TOKEN = None
if not TOKEN:
    TOKEN = getpass.getpass('Вставьте токен от @BotFather: ').strip()
    try:
        from google.colab import userdata
        userdata.set('LILITH_TELEGRAM_TOKEN', TOKEN)
        print('✅ Токен сохранён в секреты Colab — больше вводить не нужно')
    except Exception:
        print('⚠️ Не удалось сохранить в секреты — токен будет введён заново в следующий раз')
assert TOKEN, 'Токен не введён!'
print('Токен принят ✅')


In [ ]:
# 2. Скачиваем проект и ставим зависимости (~2-3 минуты)
import os
if not os.path.exists('/content/project-lady'):
    !git clone https://github.com/samagon90/project-lady.git /content/project-lady
os.chdir('/content/project-lady')
# Берём ПОСЛЕДНЮЮ версию проекта (всегда актуальную)
!git fetch --tags --force 2>/dev/null
!git checkout -f $(git describe --tags $(git rev-list --tags --max-count=1)) 2>/dev/null || true
!git reset --hard 2>/dev/null || true
!grep 'APP_VERSION' src/config.py | head -1
!pip install -q -e . 2>&1 | tail -1
print("Проект готов ✅ (если версия ниже v1.8.7 — проверьте интернет и перезапустите ячейку)")


In [ ]:
# 3. Устанавливаем Ollama (мозг) и скачиваем модель (ЛОКАЛЬНО)
import os, subprocess, time, shutil
!apt-get install -y -q zstd 2>&1 | tail -1
!curl -fsSL https://ollama.com/install.sh | sh
os.environ['PATH'] = '/usr/local/bin:' + os.environ.get('PATH', '')
if shutil.which('ollama') is None:
    print('❌ Ollama не установилась — повторяю')
    !curl -fsSL https://ollama.com/install.sh | sh
if shutil.which('ollama') is None:
    raise SystemExit('Ollama не установлена')
!nohup ollama serve > /content/ollama.log 2>&1 &
time.sleep(8)
for _ in range(15):
    r = subprocess.run(['ollama', 'list'], capture_output=True, text=True)
    if r.returncode == 0:
        print('✅ Ollama работает!')
        break
    time.sleep(4)
import psutil
ram_gb = psutil.virtual_memory().total // (1024**3)
if ram_gb < 10:
    model = 'huihui_ai/qwen3-abliterated:8b'
else:
    model = 'huihui_ai/qwen3-abliterated:14b'
print(f'RAM {ram_gb} ГБ -> модель {model}')
!ollama pull {model}
r = subprocess.run(['ollama', 'pull', 'nomic-embed-text'], capture_output=True, text=True)
if r.returncode != 0:
    print('⚠️ nomic-embed-text не скачалась — память будет упрощённой, разговор работает')
# Проверяем, что модель для разговора РЕАЛЬНО установилась
r = subprocess.run(['ollama', 'list'], capture_output=True, text=True)
installed = r.stdout if r.returncode == 0 else ''
if model.split(':')[0] in installed:
    print(f'✅ Модель установлена: {model}')
else:
    print(f'❌ Модель {model} НЕ установлена! Ollama отвечает:')
    print(installed[-500:] or '(ollama list не работает)')
# Запоминаем имя модели для ячейки 4 (.env) — без этого бот ищет qwen2.5:7b
open('/content/.lilith_model', 'w').write(model)
print('Ollama и модели готовы ✅')


In [ ]:
# 4. Настраиваем .env бота
import os
os.chdir('/content/project-lady')
# Имя модели для разговора — из ячейки 3 (обязательно! без LLM_MODEL
# бот ищет qwen2.5:7b, которой нет, и «языковая модель не работает»)
try:
    model = open('/content/.lilith_model').read().strip()
except OSError:
    import psutil
    ram_gb = psutil.virtual_memory().total // (1024**3)
    model = 'huihui_ai/qwen3-abliterated:8b' if ram_gb < 10 else 'huihui_ai/qwen3-abliterated:14b'
env = f"""TELEGRAM_TOKEN={TOKEN}
LLM_PROVIDER=ollama
LLM_BASE_URL=http://127.0.0.1:11434
LLM_MODEL={model}
LLM_TIMEOUT_SECONDS=300
LLM_RETRIES=1
EMBEDDING_MODEL=nomic-embed-text
TTS_ENABLED=false
COMFYUI_BASE_URL=http://127.0.0.1:8188
COMFYUI_CHECKPOINT=UnstableDiffusion_ema_pruned.safetensors
COMFYUI_NSFW_CHECKPOINT=UnstableDiffusion_ema_pruned.safetensors
DATA_DIR=/content/project-lady/data
TEMP_DIR=/content/project-lady/data/tmp
IMAGE_PHOTO_RATE_LIMIT_MINUTES=0
COMFYUI_NSFW_EXTREME=true
"""
open('.env','w').write(env)
!mkdir -p data/tmp
print(f".env настроен ✅ (модель: {model})")

In [ ]:
# 5. Устанавливаем Piper (голос) — опционально, если нужно голосовое
!apt-get install -y -q ffmpeg 2>&1 | tail -1
!pip install -q piper-tts 2>&1 | tail -1
!mkdir -p /content/project-lady/models/piper
import os
voice = '/content/project-lady/models/piper/ru_RU-irina-medium.onnx'
if not os.path.exists(voice):
    !curl -L -o "{voice}" "https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/ru/ru_RU/irina/medium/ru_RU-irina-medium.onnx"
    !curl -L -o "{voice}.json" "https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/ru/ru_RU/irina/medium/ru_RU-irina-medium.onnx.json"
print("Голос готов (если скачался) ✅")

### 5.5. Telegram Mini App
Туннель для Mini App поднимется **автоматически при запуске бота** (ячейка 8) —
прямо перед стартом, чтобы адрес был свежим и не успел отвалиться.

In [ ]:
# 6. Устанавливаем ComfyUI (картинки) на GPU
import os, getpass
if not os.path.exists('/content/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
!pip install -q -r /content/ComfyUI/requirements.txt 2>&1 | tail -1
ckpt_dir = '/content/ComfyUI/models/checkpoints'
os.makedirs(ckpt_dir, exist_ok=True)
ckpt = f'{ckpt_dir}/UnstableDiffusion_ema_pruned.safetensors'
# Опционально: API-ключ civitai (вставьте, если есть — скачивание надёжнее)
# Вводится один раз, в файл ноутбука НЕ сохраняется.
try:
    from google.colab import userdata
    civitai_token = userdata.get('LILITH_CIVITAI_TOKEN') or ''
    if civitai_token:
        print('✅ API-ключ civitai найден в секретах Colab')
except Exception:
    civitai_token = ''
if not civitai_token:
    civitai_token = getpass.getpass('API-ключ civitai.com (необязательно, Enter чтобы пропустить): ').strip()
    if civitai_token:
        try:
            from google.colab import userdata
            userdata.set('LILITH_CIVITAI_TOKEN', civitai_token)
            print('✅ Ключ civitai сохранён в секреты Colab')
        except Exception:
            pass
# Модель должна быть НАСТОЯЩЕЙ и БОЛЬШОЙ (~2 ГБ). Если файл пустой
# или битый — удаляем и пробуем следующий источник.
def valid_ckpt(path):
    try:
        return os.path.exists(path) and os.path.getsize(path) > 1500 * 1024 * 1024
    except OSError:
        return False
sources = []
if civitai_token:
    sources.append(('civitai.com Unstable Diffusion NSFW (с вашим API-ключом)', f'https://civitai.com/api/download/models/91623?token={civitai_token}'))
sources += [
    ('civitai.com Unstable Diffusion NSFW', 'https://civitai.com/api/download/models/91623'),
    ('HuggingFace lllyasviel', 'https://huggingface.co/lllyasviel/fav_models/resolve/main/fav/majicmixRealistic_v7.safetensors'),
    ('HuggingFace digiplay', 'https://huggingface.co/digiplay/majicMIX_realistic_v7/resolve/main/majicmixRealistic_v7.safetensors'),
    ('Stable Diffusion 1.5 (надёжный запасной)', 'https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors'),
]
ok = valid_ckpt(ckpt)
if not ok:
    if os.path.exists(ckpt):
        print('⚠️ Файл модели битый/пустой — удаляю и качаю заново')
        os.remove(ckpt)
    for name, url in sources:
        print(f'Скачиваю {name}... (~2 ГБ)')
        !curl -L --fail --max-time 3600 -o "{ckpt}" "{url}"
        if valid_ckpt(ckpt):
            print(f'✅ Модель скачана ({name}):', os.path.getsize(ckpt)//(1024**3), 'ГБ')
            ok = True
            break
        else:
            print('⚠️ Не получилось — файл маленький/битый, пробую следующий')
            if os.path.exists(ckpt):
                os.remove(ckpt)
if not ok:
    raise SystemExit('❌ Не удалось скачать модель. Проверьте интернет или скачайте вручную.')
print('ComfyUI готов ✅')

# Устанавливаем IPAdapter (твёрдый референс аватара Лилит)
!git clone https://github.com/cubiq/ComfyUI_IPAdapter_plus.git /content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus 2>/dev/null || echo 'IPAdapter уже есть'
!pip install -q insightface onnxruntime 2>&1 | tail -1
os.makedirs('/content/ComfyUI/models/ipadapter', exist_ok=True)
# Скачиваем модель IPAdapter (plus, ~2.5 ГБ) — для SD1.5
ipa = '/content/ComfyUI/models/ipadapter/ip-adapter-plus_sd15.safetensors'
if not os.path.exists(ipa):
    !curl -L --fail --max-time 3600 -o "{ipa}" "https://huggingface.co/h94/IP-Adapter/resolve/main/models/ip-adapter-plus_sd15.safetensors"
print('IPAdapter готов ✅')
# Копируем аватар Лилит в input ComfyUI (референс)
os.makedirs('/content/ComfyUI/input', exist_ok=True)
!cp /content/project-lady/assets/emotions/lilith_playful.png /content/ComfyUI/input/lilith_ref.png 2>/dev/null || echo 'аватар не найден'
print('Референс Лилит скопирован ✅')

# Устанавливаем AnimateDiff (видео) + VideoHelperSuite (mp4)
!git clone https://github.com/Kosinkadink/ComfyUI-AnimateDiff-Evolved.git /content/ComfyUI/custom_nodes/ComfyUI-AnimateDiff-Evolved 2>/dev/null || echo 'AnimateDiff уже есть'
!git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git /content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite 2>/dev/null || echo 'VHS уже есть'
!pip install -q -r /content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite/requirements.txt 2>&1 | tail -1
os.makedirs('/content/ComfyUI/models/animatediff_models', exist_ok=True)
motion = '/content/ComfyUI/models/animatediff_models/mm_sd_v15_v2.ckpt'
if not os.path.exists(motion):
    !curl -L --fail --max-time 3600 -o "{motion}" "https://huggingface.co/guoyww/animatediff/resolve/main/mm_sd_v15_v2.ckpt"
print('AnimateDiff готов ✅ (видео /video)')


In [ ]:
# 7. Запускаем ComfyUI на GPU (в фоне)
import subprocess, time, urllib.request
log = open('/content/comfyui.log','w')
proc = subprocess.Popen(['python','/content/ComfyUI/main.py','--listen','127.0.0.1','--port','8188'], stdout=log, stderr=log)
print("ComfyUI запускается...")
for _ in range(60):
    time.sleep(2)
    try:
        urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=2)
        print("✅ ComfyUI работает на GPU!")
        break
    except Exception:
        pass
else:
    print("ComfyUI не поднялся — смотрите лог:")
    print(open('/content/comfyui.log').read()[-2000:])

In [ ]:
# 8. Запускаем бота + Mini App туннель (всё само)
import os, subprocess, time, re, shutil
os.chdir('/content/project-lady')
os.environ.setdefault('PATH', '/usr/local/bin:' + os.environ.get('PATH', ''))

# --- Mini App: туннель ПЕРЕД запуском бота (свежий адрес) ---
if shutil.which('cloudflared') is None:
    subprocess.run(['curl', '-L', '-o', '/usr/local/bin/cloudflared', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], capture_output=True)
    subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], capture_output=True)

def find_url_in_file(path):
    try:
        txt = open(path, encoding='utf-8', errors='replace').read()
    except OSError:
        return None
    m = re.search(r'https://[a-z0-9-]+\.(?:loca\.lt|trycloudflare\.com)', txt)
    return m.group(0) if m else None

tunnel_url = None
print('Поднимаю туннель Mini App (порт 8001)...')
# Запускаем ОБА туннеля сразу (localtunnel + cloudflared), пишем в лог-файлы
with open('/content/lt.log', 'w') as f:
    subprocess.Popen(['npx', '-y', 'localtunnel', '--port', '8001'], stdout=f, stderr=subprocess.STDOUT)
with open('/content/cf.log', 'w') as f:
    subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8001'], stdout=f, stderr=subprocess.STDOUT)

# Опрашиваем логи до 3 минут (неблокирующе) — кто первый даст URL
for _ in range(90):
    # Сначала cloudflared: у него НЕТ страницы-предупреждения (localtunnel
    # иногда показывает заглушку «ваш IP…», из-за которой Mini App не открывается)
    tunnel_url = find_url_in_file('/content/cf.log') or find_url_in_file('/content/lt.log')
    if tunnel_url:
        break
    time.sleep(2)

if tunnel_url:
    env_path = '/content/project-lady/.env'
    env = open(env_path).read()
    if 'MINIAPP_HOST' not in env:
        env += '\nMINIAPP_HOST=0.0.0.0\nMINIAPP_PORT=8001\n'
    lines = [l for l in env.splitlines() if not l.startswith('WEBAPP_URL=')]
    lines.append(f'WEBAPP_URL={tunnel_url}')
    open(env_path, 'w').write('\n'.join(lines) + '\n')
    print(f'✅ WEBAPP_URL записан: {tunnel_url}')
else:
    print('⚠️ Туннель не дал адрес за 3 мин. Mini App можно настроить позже (бот работает).')

# --- Миграции и запуск бота ---
!python -m alembic upgrade head 2>&1 | tail -1
# Проверяем, что Ollama жива (локальная модель) — иначе перезапускаем
if shutil.which('ollama') is not None:
    r = subprocess.run(['ollama', 'list'], capture_output=True, text=True)
    if r.returncode != 0:
        print('Ollama не запущена — запускаю заново...')
        !nohup ollama serve > /content/ollama.log 2>&1 &
        time.sleep(10)
else:
    print('⚠️ ollama не установлена — разговор работать НЕ будет. Запустите ячейку 3.')
bot_proc = subprocess.Popen(['python', '-m', 'src.main'], stdout=open('/content/bot.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(10)
log = open('/content/bot.log').read()
print(log[-2000:])
print('\n✅ Бот запущен! Идите в Telegram и напишите /start. Mini App: /app')

# --- САМОПРОВЕРКА: что реально работает, а что нет ---
import json as _json
import urllib.request

def test_llm():
    try:
        env = {}
        for line in open('/content/project-lady/.env').read().splitlines():
            if '=' in line:
                k, v = line.split('=', 1)
                env[k] = v
        model = env.get('LLM_MODEL', '?')
        payload = _json.dumps({
            'model': model,
            'messages': [{'role': 'user', 'content': 'Ответь одним словом: работаешь?'}],
            'stream': False,
            'options': {'num_predict': 15},
        }).encode()
        req = urllib.request.Request('http://127.0.0.1:11434/api/chat', data=payload,
                                     headers={'Content-Type': 'application/json'})
        r = urllib.request.urlopen(req, timeout=240)
        reply = _json.loads(r.read()).get('message', {}).get('content', '')
        return True, reply.strip()[:80]
    except Exception as e:
        return False, str(e)[:160]

def test_comfy():
    try:
        urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=5)
        return True, ''
    except Exception as e:
        return False, str(e)[:160]

def test_checkpoint():
    import glob
    ckpts = glob.glob('/content/ComfyUI/models/checkpoints/*.safetensors')
    big = [f for f in ckpts if os.path.getsize(f) > 500 * 1024 * 1024]
    if big:
        return True, ', '.join(os.path.basename(f) for f in big)
    if ckpts:
        return False, 'файлы слишком маленькие: ' + ', '.join(os.path.basename(f) for f in ckpts)
    return False, 'папка checkpoints пуста'

ok_llm, err_llm = test_llm()
ok_cf, err_cf = test_comfy()
ok_ckpt, info_ckpt = test_checkpoint()
print('\n========== САМОПРОВЕРКА ==========')
print('✅ Языковая модель отвечает:' if ok_llm else '❌ Языковая модель НЕ отвечает')
if ok_llm:
    print('   Ответ модели:', repr(err_llm))
else:
    print('   Ошибка:', err_llm)
    print('   💡 Проверьте ячейку 3 (Ollama + qwen3-abliterated) и .env (LLM_MODEL)')
print('✅ ComfyUI работает (картинки):' if ok_cf else '❌ ComfyUI НЕ работает (картинки)')
if not ok_cf:
    print('   Ошибка:', err_cf)
    print('   💡 Проверьте ячейки 6-7 (запуск ComfyUI)')
print('✅ Checkpoint для картинок (~2 ГБ):' if ok_ckpt else '❌ Checkpoint для картинок НЕ найден')
print('   ', info_ckpt)
if not ok_ckpt:
    print('   💡 Перезапустите ячейку 7 — она скачает модель (~2 ГБ)')
if ok_llm and ok_cf and ok_ckpt:
    print('🎉 Всё работает!')
elif not ok_llm and not ok_cf:
    print('💡 Не работает и разговор, и картинки — чаще всего это старая версия ноутбука. Скачайте новый:')
    print('   https://github.com/samagon90/project-lady/archive/refs/tags/v1.8.7.zip → колаб → загрузить блокнот')


## 📋 Шпаргалка

- **Остановить бота**: Среда выполнения → Прервать.
- **Перезапустить после обрыва**: откройте ноутбук → Среда выполнения → Выполнить всё → вставить токен → готово.
- **Логи**: `print(open('/content/bot.log').read()[-2000:])` — вставьте в новую ячейку и запустите.
- **Сменить модель**: в ячейке 3 поменяйте `model = ...` на `dolphin-llama3:8b` или `qwen2.5:7b`.

## ⚠️ Важно
- Без карты и бесплатно — это лучший вариант, но сессии не вечные. Для 24/7 нужен платный VPS или карта (Oracle).
- Если Colab пишет про лимит GPU — подождите час или используйте CPU-среду (бот будет работать, картинки медленные).
- Не закрывайте вкладку с ноутбуком, пока бот нужен.